In [3]:
# import libraries
from delta.tables import *
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.types import DateType, StringType, TimestampType, DecimalType
from pyspark.sql.functions import current_timestamp, col
from dotenv import dotenv_values

In [4]:
aws_credentials = dotenv_values("./env")

In [5]:
spark = SparkSession \
    .builder \
    .appName("etl-enriched-users-analysis-py") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", aws_credentials['ACCESS_KEY_ID']) \
    .config("spark.hadoop.fs.s3a.secret.key", aws_credentials['SECRET_ACCESS_KEY']) \
    .config("spark.hadoop.fs.s3a.path.style.access", True) \
    .config("spark.hadoop.fs.s3a.fast.upload", True) \
    .config("spark.hadoop.fs.s3a.multipart.size", 104857600) \
    .config("fs.s3a.connection.maximum", 100) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 03:22:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/25 03:22:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
# show configured parameters
print(SparkConf().getAll())

[('spark.eventLog.enabled', 'true'), ('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false'), ('spark.hadoop.fs.s3a.path.style.access', 'true'), ('spark.eventLog.dir', 'file:/opt/bitnami/spark/logs/events'), ('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'), ('spark.app.name', 'etl-enriched-users-analysis-py'), ('spark.history.fs.logDirectory', 'file:/opt/bitnami/spark/logs/events'), ('spark.executor.memory', '1g'), ('spark.master', 'spark://spark-master:7077'), ('spark.history.provider', 'org.apache.spark.deploy.history.FsHistoryProvider'), ('spark.hadoop.fs.s3a.access.key', 'minioadmin'), ('spark.submit.deployMode', 'client'), ('spark.delta.logStore.class', 'org.apache.spark.sql.delta.storage.S3SingleDriverLogStore'), ('spark.driver.extraJavaOptions', '--add-exports java.base/sun.nio.ch=ALL-UNNAMED'), ('spark.hadoop.fs.s3a.secret.key', 'minioadmin'), ('spark.hadoop.fs.s3a.multipart.size', '104857600'), ('spark.driver.host', 'jup

In [7]:
# set log level
#spark.sparkContext.setLogLevel("INFO")

In [8]:
# get_subscription_file = "s3a://landing/subscription/*.json"
get_device_file = "s3a://landing/*.json"

In [9]:
# json file from landing zone
df_device = spark.read \
    .format("json") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .json(get_device_file)

26/02/25 03:22:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [10]:
# get number of partitions
print(df_device.rdd.getNumPartitions())

2


In [11]:
# count amount of rows ingested from lake
df_device.count()

200

In [40]:
# show
df_device.show()

+------------+--------------------+----+------------+-------------------+-----------------+--------------------+--------------------+-------+-------+
|build_number|dt_current_timestamp|  id|manufacturer|              model|         platform|       serial_number|                 uid|user_id|version|
+------------+--------------------+----+------------+-------------------+-----------------+--------------------+--------------------+-------+-------+
|         100|       1654630947030|2392|        Acer|         OnePlus 6T|    Windows Phone|UVr864F8zUbyYOAUd...|f622857e-f179-4f4...|   5562|     50|
|         158|       1654630947030|6874|          HP|  Google Pixel 3 XL|Windows 10 Mobile|pEekWH7zGxVITv6NT...|9bb2c744-a317-421...|   9062|    788|
|         114|       1654630947030|9767|       Apple|          iPhone SE|        Danger OS|05skEogwZlX7j6twhhXX|abc61202-8853-48c...|    176|    398|
|         442|       1654630947030|1421|        Dell|          iPhone SE|              iOS|VMTnd2mMQ

In [41]:
# show
df_device.printSchema()

root
 |-- build_number: long (nullable = true)
 |-- dt_current_timestamp: long (nullable = true)
 |-- id: long (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- uid: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- version: long (nullable = true)



In [42]:
# write parquet to minio
df_device.write \
    .mode("overwrite") \
    .parquet("s3a://bronze/device/parquet")

In [43]:
# write delta to minio
df_device.write \
    .mode("overwrite") \
    .format("delta") \
    .save("s3a://bronze/device/delta")

In [44]:
lk_orders_path = "s3a://bronze/device/delta"
delta_table = DeltaTable.forPath(spark, lk_orders_path)

print("📋 Transaction History:")
history_df = delta_table.history()
history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    "operationMetrics"
).show(10, truncate=False)

📋 Transaction History:
+-------+-------------------+---------+--------------------------------------+--------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                   |operationMetrics                                              |
+-------+-------------------+---------+--------------------------------------+--------------------------------------------------------------+
|3      |2026-02-25 02:51:47|WRITE    |{mode -> Overwrite, partitionBy -> []}|{numFiles -> 2, numOutputRows -> 200, numOutputBytes -> 20810}|
|2      |2026-02-25 02:43:50|WRITE    |{mode -> Overwrite, partitionBy -> []}|{numFiles -> 2, numOutputRows -> 200, numOutputBytes -> 20810}|
|1      |2026-02-25 02:39:56|WRITE    |{mode -> Overwrite, partitionBy -> []}|{numFiles -> 2, numOutputRows -> 200, numOutputBytes -> 20810}|
|0      |2026-02-25 02:25:12|WRITE    |{mode -> Overwrite, partitionBy -> []}|{numFiles -> 2, numOutputRows -> 200, numOutput

In [46]:
print("\n🔍 Latest Transaction Details:")
latest_version = history_df.select("version").first()[0]


🔍 Latest Transaction Details:


In [47]:
print(f"Latest version: {latest_version}")

Latest version: 3


In [48]:
print("\n📁 Table Details:")
spark.sql(f"DESCRIBE DETAIL delta.`{lk_orders_path}`") \
    .select("location", "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion") \
    .show(truncate=False)


📁 Table Details:
+-------------------------+--------+-----------+----------------+----------------+
|location                 |numFiles|sizeInBytes|minReaderVersion|minWriterVersion|
+-------------------------+--------+-----------+----------------+----------------+
|s3a://bronze/device/delta|2       |20810      |1               |2               |
+-------------------------+--------+-----------+----------------+----------------+



In [49]:
reader_snapshot = spark.read.format("delta").load(lk_orders_path)
initial_count = reader_snapshot.count()
print(f"📈 Reader sees {initial_count} users")

📈 Reader sees 200 users


In [50]:
spark.stop()